# Probe training on steered and unsteered hive states

This is one executable experiment. It starts with the existing continued
Qwen3.5-0.8B bee adapter and the saved layer-19/20 vector probes. The old probes
are used **only to generate a steered source environment**. For each of three
source seeds, baseline and old equal-normalized steering each run for two
simulated hours. A new 14-output probe is then fitted to the local observations,
chosen actions, and immediate hive-vector effects from both environments and
both hours. Teacher-forced feature extraction uses neutral model activations
for every source example, keeping the new probe's activation coordinates
comparable across the two environment types.

The notebook screens all text decoder layers, reports probe accuracy separately
by hour and environment on held-out source examples, and constructs new
individual and equal-normalized attribute directions. Separate two-hour
calibration runs learn nonnegative combination weights from measured hive
outcomes. Finally, four fresh seeds compare baseline, equal-weight steering,
and learned-weight steering for two simulated hours each. Every bee sees only
its local view and feasible-action menu. Every action receives a timestamped
14-component researcher-only vector snapshot.

The probe target is still the **immediate change after one action**. The new
training distribution includes states reached under steering; this notebook
does not turn the probe into a future-action outcome predictor.

In [ ]:
%pip install --upgrade --quiet unsloth unsloth_zoo "transformers>=5.0.0" "trl>=0.27.0" datasets accelerate peft scikit-learn

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import copy, json, math, sys, time, zipfile
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import f1_score
from unsloth import FastLanguageModel

INPUT_ROOT=Path('/kaggle/input')
runtime_archive=next(INPUT_ROOT.rglob('beehive_qwen35_runtime_bundle.zip'),None)
menu_module=next(INPUT_ROOT.rglob('bee_menu_qwen_policy.py'))
probe_files={layer:next(INPUT_ROOT.rglob(f'probe_layer_{layer}.npz')) for layer in (19,20)}
adapter_files=[p for p in INPUT_ROOT.rglob('adapter_model.safetensors')
               if 'menu' in str(p.parent).lower()]
if not adapter_files: raise FileNotFoundError('Attach the private continued menu-LoRA adapter dataset')
ADAPTER_DIR=adapter_files[0].parent
RUN_ROOT=Path('/kaggle/working')/('bee_mixed_probe_'+datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ'))
RUN_ROOT.mkdir(parents=True,exist_ok=False)
if runtime_archive is not None:
    RUNTIME_DIR=RUN_ROOT/'runtime'
    with zipfile.ZipFile(runtime_archive) as z: z.extractall(RUNTIME_DIR)
else:
    RUNTIME_DIR=next(INPUT_ROOT.rglob('beehive_sim.py')).parent
sys.path[:0]=[str(menu_module.parent),str(RUNTIME_DIR)]
from beehive_sim import Config, World, Simulation, VECTOR_NAMES, weighted_research_score, validate_action
from bee_menu_qwen_policy import BeeMenuQwenPolicy, MENU_SYSTEM_PROMPT
from bee_feasibility import feasible_actions
SOURCE_META=json.loads(next(INPUT_ROOT.rglob('probe_source_metadata.json')).read_text())
SOURCE_CONFIG=dict(SOURCE_META['calibration']['config'])
assert len(VECTOR_NAMES)==14
print('Adapter',ADAPTER_DIR,'probe files',probe_files,'output',RUN_ROOT)

In [ ]:
SMOKE=False
BATCH_SIZE=16
MAX_NEW_TOKENS=160
SOURCE_SEEDS=(90,91,92)
SOURCE_HOURS=0.10 if SMOKE else 2.0
CAL_SEEDS=(93,) if SMOKE else (93,94)
CAL_HOURS=0.10 if SMOKE else 2.0
FINAL_SEEDS=(100,) if SMOKE else (100,101,102,103)
FINAL_HOURS=0.10 if SMOKE else 2.0
FRACTION=0.03
POSITION='all'
BOOTSTRAP_LAYERS=(19,20)
SELECTED_LAYERS=BOOTSTRAP_LAYERS

model,processor=FastLanguageModel.from_pretrained(
    model_name=str(ADAPTER_DIR),max_seq_length=4096,dtype=None,
    load_in_4bit=False,load_in_16bit=True,full_finetuning=False)
FastLanguageModel.for_inference(model)
model.eval()

def decoder_candidates(model):
    found=[]
    for name,module in model.named_modules():
        if not isinstance(module,nn.ModuleList) or len(module)<8: continue
        if not any(hasattr(module[0],field) for field in ('self_attn','linear_attn','mlp')): continue
        score=len(module)+100*('language_model' in name)+50*('text' in name)
        score-=100*any(word in name for word in ('visual','vision','image'))
        found.append((score,name,module))
    return sorted(found,reverse=True)
_,LAYER_PATH,DECODER_LAYERS=decoder_candidates(model)[0]
N_LAYERS=len(DECODER_LAYERS)
assert N_LAYERS>max(BOOTSTRAP_LAYERS)

def normalized(vector):
    vector=np.asarray(vector,dtype=np.float32)
    size=float(np.linalg.norm(vector))
    assert np.isfinite(size) and size>1e-10
    return vector/size

old_W={layer:np.load(probe_files[layer])['W'].astype(np.float32)
       for layer in BOOTSTRAP_LAYERS}
old_norms={layer:np.linalg.norm(old_W[layer],axis=0) for layer in BOOTSTRAP_LAYERS}
old_active=np.flatnonzero(np.all(np.stack(
    [old_norms[layer]>1e-10 for layer in BOOTSTRAP_LAYERS]),axis=0))
assert old_active.tolist()==[2,3,8,10,11,12]
BOOTSTRAP_DIRS={layer:normalized((old_W[layer][:,old_active]/
                                  old_norms[layer][old_active]).sum(axis=1))
                for layer in BOOTSTRAP_LAYERS}
print('Text decoder',LAYER_PATH,N_LAYERS,'layers; bootstrap columns',old_active.tolist(),
      'smoke',SMOKE)

## 1. Run baseline and steered source environments over both hours

The initial steering only changes how source worlds evolve. It uses the old
equal-normalized direction at the previously tested 0.03 strength. Each bee's
model still receives its local observation and menu. The first baseline run
measures fixed activation norms; the same norms are used for all source steered
runs. Full runs have 400 actions per source arm, covering both simulated hours.
The smoke run caps each arm at 24 actions to catch code and data errors quickly.

In [ ]:
def config_for(seed,hours):
    data=dict(SOURCE_CONFIG)
    data.update(seed=int(seed),duration_hours=float(hours),
                max_actions=int(data['n_bees']*math.ceil(hours*60/data['round_minutes'])),
                model='Qwen3.5-0.8B continued menu LoRA; mixed-state probe experiment')
    if SMOKE: data['max_actions']=min(data['max_actions'],24)
    return Config(**data)

class CapturingPolicy(BeeMenuQwenPolicy):
    def __init__(self,*args,capture=False,**kwargs):
        super().__init__(*args,**kwargs)
        self.capture=bool(capture); self.observations=[]
    def decide_batch(self,observations):
        if self.capture: self.observations.extend(copy.deepcopy(observations))
        return super().decide_batch(observations)

class NormCapturePolicy(CapturingPolicy):
    def __init__(self,*args,**kwargs):
        super().__init__(*args,**kwargs)
        self.samples={layer:[] for layer in SELECTED_LAYERS}
    def decide_batch(self,observations):
        handles=[]
        try:
            for layer_i in SELECTED_LAYERS:
                def capture_norm(_module,_inputs,output,layer_i=layer_i):
                    if len(self.samples[layer_i])>=256: return
                    hidden=output[0] if isinstance(output,(tuple,list)) else output
                    values=hidden[:,-1,:].float().norm(dim=-1).detach().cpu().tolist()
                    self.samples[layer_i].extend(values[:256-len(self.samples[layer_i])])
                handles.append(DECODER_LAYERS[layer_i].register_forward_hook(capture_norm))
            return super().decide_batch(observations)
        finally:
            for handle in handles: handle.remove()

activation_norms={}
class ColumnSteeredPolicy(CapturingPolicy):
    def __init__(self,directions,capture=False):
        super().__init__(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
        self.directions=directions; self.hook_calls=0
        self.label='mixed_probe_column_steered_menu'
    def decide_batch(self,observations):
        handles=[]
        try:
            for layer_i,unit in self.directions.items():
                size=FRACTION*activation_norms[layer_i]/math.sqrt(len(self.directions))
                cached={}
                def steer(_module,_inputs,output,unit=unit,size=size,cached=cached):
                    hidden=output[0] if isinstance(output,(tuple,list)) else output
                    key=(str(hidden.device),hidden.dtype)
                    if key not in cached:
                        cached[key]=torch.as_tensor(unit,device=hidden.device,dtype=hidden.dtype).view(1,1,-1)*size
                    shift=cached[key]
                    assert hidden.shape[-1]==shift.shape[-1]
                    self.hook_calls+=1
                    if POSITION=='response' and hidden.shape[1]>1:
                        altered=hidden.clone(); altered[:,-1:,:]+=shift
                    else:
                        altered=hidden+shift
                    if isinstance(output,tuple): return (altered,*output[1:])
                    if isinstance(output,list): return [altered,*output[1:]]
                    return altered
                handles.append(DECODER_LAYERS[layer_i].register_forward_hook(steer))
            result=super().decide_batch(observations)
            assert self.hook_calls>0
            return result
        finally:
            for handle in handles: handle.remove()

def run_arm(name,seed,hours,directions=None,capture=False,capture_norm=False):
    world=World(config_for(seed,hours))
    if capture_norm:
        policy=NormCapturePolicy(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
    elif directions is None:
        policy=CapturingPolicy(model,processor,MAX_NEW_TOKENS,BATCH_SIZE,capture=capture)
    else:
        policy=ColumnSteeredPolicy(directions,capture=capture)
    output_dir=RUN_ROOT/name
    sim=Simulation(world,policy,output_dir=output_dir)
    transcript=sim.run(progress_every=200,decision_batch_size=BATCH_SIZE)
    audit=sim.audit; outcomes=Counter(r['outcome']['status'] for r in audit)
    initial=sim.baseline; final=transcript[-1]['vector']
    immediate=float(sum(np.mean(r['immediate_action_delta']) for r in audit))
    background=float(sum(np.mean(r['background_delta']) for r in audit))
    row={'seed':int(seed),'name':name,'hours':hours,'actions':len(audit),
         'applied':outcomes['applied'],'rejected':outcomes['rejected'],
         'schema_invalid':sum(validate_action(r['action']) is not None for r in audit),
         'initial_vector':initial,'final_vector':final,
         'initial_score':weighted_research_score(initial,[1.0]*14),
         'final_score':weighted_research_score(final,[1.0]*14),
         'immediate_score_change':immediate,'background_score_change':background,
         'hook_calls':getattr(policy,'hook_calls',0),'output_dir':str(output_dir)}
    assert len(audit)==config_for(seed,hours).max_actions
    assert abs(row['initial_score']+immediate+background-row['final_score'])<1e-6
    assert all(len(x['vector'])==14 and all(np.isfinite(x['vector'])) and
               all(0<=v<=1 for v in x['vector']) for x in transcript)
    if directions is not None: assert row['hook_calls']>0
    if capture: assert len(policy.observations)==len(audit)
    print(name,'score',round(row['final_score'],6),'applied',row['applied'],
          'rejected',row['rejected'],flush=True)
    return row,policy

def action_list(row):
    return [json.loads(line)['action'] for line in
            (Path(row['output_dir'])/'actions.jsonl').open()]

def annotate_changes(row,baseline):
    left,right=action_list(row),action_list(baseline)
    assert len(left)==len(right)
    row['changed_actions_vs_baseline']=sum(a!=b for a,b in zip(left,right))
    row['score_difference_vs_baseline']=row['final_score']-baseline['final_score']
    row['vector_difference_vs_baseline']=(np.asarray(row['final_vector'])-
                                          np.asarray(baseline['final_vector'])).tolist()
    return row

In [ ]:
source_runs=[]; source_summaries=[]
for seed in SOURCE_SEEDS:
    baseline,policy=run_arm(f'source_{seed}_baseline',seed,SOURCE_HOURS,
                            capture=True,capture_norm=(seed==SOURCE_SEEDS[0]))
    if seed==SOURCE_SEEDS[0]:
        activation_norms.update({layer:float(np.median(policy.samples[layer]))
                                 for layer in BOOTSTRAP_LAYERS})
        assert all(np.isfinite(v) and v>0 for v in activation_norms.values())
        print('Bootstrap activation norms',activation_norms)
    baseline['arm']='baseline'; baseline['changed_actions_vs_baseline']=0
    baseline['score_difference_vs_baseline']=0.0
    baseline['vector_difference_vs_baseline']=[0.0]*14
    source_runs.append((baseline,policy));source_summaries.append(baseline)
    steered,policy=run_arm(f'source_{seed}_bootstrap_equal',seed,SOURCE_HOURS,
                           BOOTSTRAP_DIRS,capture=True)
    steered['arm']='bootstrap_equal'; annotate_changes(steered,baseline)
    source_runs.append((steered,policy));source_summaries.append(steered)
source_changed=sum(r['changed_actions_vs_baseline'] for r in source_summaries)
if not SMOKE: assert source_changed>0,'Bootstrap steering did not diversify source states'
print('Source actions',sum(r['actions'] for r in source_summaries),
      'bootstrap changed actions',source_changed)
pd.DataFrame([{k:v for k,v in row.items() if k not in
              ('initial_vector','final_vector','vector_difference_vs_baseline')}
              for row in source_summaries]).to_csv(RUN_ROOT/'source_run_summary.csv',index=False)

## 2. Extract neutral response activations and fit the 14-output probe

The probe input is the model's response representation given the current
local observation and the action that the source policy chose. Source states
come from both baseline and steered worlds, in both hour windows. The probe
target is the immediate post-action minus pre-action 14-vector. Source seeds
90 and 91 train the probe. Seed 92 supplies alternating events for layer
selection and a separate report split; each split covers both environment
types and both hours. A fixed 0.5 change-mask threshold makes predicted
unchanged attributes exactly zero. The report includes hour-by-environment
metrics on the held-out events.

In [ ]:
observations=[]; action_rows=[]; source_seed_ids=[]; source_arm_ids=[]; source_hours=[]
for summary,policy in source_runs:
    rows=[json.loads(line) for line in
          (Path(summary['output_dir'])/'actions.jsonl').open()]
    assert len(rows)==len(policy.observations)==summary['actions']
    observations.extend(policy.observations);action_rows.extend(rows)
    source_seed_ids.extend([summary['seed']]*len(rows))
    source_arm_ids.extend([summary['arm']]*len(rows))
    per_hour=int(SOURCE_CONFIG['n_bees']*60/SOURCE_CONFIG['round_minutes'])
    source_hours.extend([1+(int(row['event_id'])-1)//per_hour for row in rows])
source_seed_ids=np.asarray(source_seed_ids)
source_arm_ids=np.asarray(source_arm_ids)
source_hours=np.asarray(source_hours)
targets=np.asarray([row['immediate_action_delta'] for row in action_rows],dtype=np.float32)
assert targets.shape==(len(observations),14)
train_pos=np.flatnonzero(np.isin(source_seed_ids,SOURCE_SEEDS[:2]))
held=np.flatnonzero(source_seed_ids==SOURCE_SEEDS[-1])
screen_pos,report_pos=held[::2],held[1::2]
assert len(train_pos)>0 and len(screen_pos)>0 and len(report_pos)>0
if not SMOKE:
    for positions in (train_pos,screen_pos,report_pos):
        assert set(source_hours[positions])=={1,2}
        assert set(source_arm_ids[positions])=={'baseline','bootstrap_equal'}
print('Probe splits',len(train_pos),len(screen_pos),len(report_pos))
print('Source condition counts',pd.DataFrame({'seed':source_seed_ids,'arm':source_arm_ids,
                                              'hour':source_hours}).value_counts().to_string())

def encoded_chat(messages,add_generation_prompt):
    enc=processor.apply_chat_template(messages,tokenize=True,
        add_generation_prompt=add_generation_prompt,return_tensors='pt',return_dict=True)
    return {k:v for k,v in enc.items() if isinstance(v,torch.Tensor)}

pooled={}; response_span=[0,0]; handles=[]
for layer_i,layer in enumerate(DECODER_LAYERS):
    def capture(_module,_inputs,output,layer_i=layer_i):
        hidden=output[0] if isinstance(output,(tuple,list)) else output
        start,end=response_span
        pooled[layer_i]=hidden[0,start:end].float().mean(dim=0).detach().cpu().numpy()
    handles.append(layer.register_forward_hook(capture))
features=None; feature_start=time.perf_counter()
try:
    for index,(obs,row) in enumerate(zip(observations,action_rows)):
        enriched=dict(obs,feasible_actions=feasible_actions(obs))
        prompt=[{'role':'system','content':MENU_SYSTEM_PROMPT},
                {'role':'user','content':json.dumps(enriched,ensure_ascii=False,
                                                  separators=(',',':'),allow_nan=False)}]
        action_text=json.dumps(row['action'],ensure_ascii=False,separators=(',',':'),allow_nan=False)
        prompt_enc=encoded_chat(prompt,True)
        full_enc=encoded_chat(prompt+[{'role':'assistant','content':action_text}],False)
        pids,fids=prompt_enc['input_ids'][0],full_enc['input_ids'][0]
        assert torch.equal(pids,fids[:len(pids)])
        response_span[:]=[len(pids),len(fids)-1]
        assert response_span[1]>response_span[0] and len(fids)<=4096
        pooled.clear()
        with torch.inference_mode():
            result=model(**{k:v.to(model.device) for k,v in full_enc.items()},use_cache=False)
        del result
        if features is None:
            features=np.empty((len(observations),N_LAYERS,len(pooled[0])),dtype=np.float32)
        for layer_i in range(N_LAYERS): features[index,layer_i]=pooled[layer_i]
        if (index+1)%100==0:
            print('Features',index+1,'/',len(observations),'minutes',
                  round((time.perf_counter()-feature_start)/60,1),flush=True)
finally:
    for handle in handles: handle.remove()
assert features is not None and np.isfinite(features).all()
print('Feature shape',features.shape)

In [ ]:
scales=[]
for j in range(14):
    nz=np.abs(targets[train_pos,j]);nz=nz[nz>1e-10]
    scales.append(max(float(np.percentile(nz,75)),1e-4) if len(nz) else 1.0)
scales=np.asarray(scales,dtype=np.float32)
y_scaled=targets/scales
row_weights=1+3*np.any(np.abs(targets[train_pos])>1e-10,axis=1).astype(np.float32)

def weighted_mae(pred,truth):
    weights=np.where(np.abs(truth)>1e-10,4.0,1.0)
    return float(np.sum(weights*np.abs(pred-truth))/weights.sum())

layer_models={};screen_rows=[]
for layer_i in range(N_LAYERS):
    x=features[train_pos,layer_i]
    mu=x.mean(axis=0);sd=np.maximum(x.std(axis=0),1e-3)
    ridge=Ridge(alpha=100.0,solver='lsqr',max_iter=1000)
    ridge.fit((x-mu)/sd,y_scaled[train_pos],sample_weight=row_weights)
    pred=ridge.predict((features[screen_pos,layer_i]-mu)/sd)
    screen_rows.append({'layer':layer_i,'weighted_scaled_mae':weighted_mae(pred,y_scaled[screen_pos])})
    layer_models[layer_i]=(ridge,mu,sd)
screen_table=pd.DataFrame(screen_rows).sort_values('weighted_scaled_mae')
SELECTED_LAYERS=tuple(screen_table.head(2).layer.astype(int).tolist())
screen_table.to_csv(RUN_ROOT/'layer_screen.csv',index=False)
print('Selected layers',SELECTED_LAYERS,'screen leaders',screen_table.head(5).to_dict('records'))

train_change_counts=np.count_nonzero(np.abs(targets[train_pos])>1e-10,axis=0)
ACTIVE=np.flatnonzero(train_change_counts>=(1 if SMOKE else 5))
assert len(ACTIVE)>=1,train_change_counts
W={};U={};probe_metrics=[];probe_controls={}
for layer_i in SELECTED_LAYERS:
    ridge,mu,sd=layer_models[layer_i]
    x_train=(features[train_pos,layer_i]-mu)/sd
    x_report=(features[report_pos,layer_i]-mu)/sd
    signed=ridge.predict(x_report)*scales
    sparse=np.zeros_like(signed)
    for j in range(14):
        changed=(np.abs(targets[train_pos,j])>1e-10).astype(int)
        if len(np.unique(changed))<2:
            prob=np.full(len(report_pos),float(changed[0]))
        else:
            clf=LogisticRegression(C=0.1,class_weight='balanced',max_iter=300,
                                   solver='liblinear',random_state=271)
            clf.fit(x_train,changed)
            prob=clf.predict_proba(x_report)[:,1]
        sparse[:,j]=np.where(prob>=0.5,signed[:,j],0)
    W[layer_i]=(ridge.coef_.T/sd[:,None])*scales[None,:]
    bias=(ridge.intercept_-(mu/sd)@ridge.coef_.T)*scales
    norms=np.linalg.norm(W[layer_i][:,ACTIVE],axis=0)
    assert np.all(np.isfinite(norms)) and np.all(norms>1e-10)
    U[layer_i]=W[layer_i][:,ACTIVE]/norms
    np.savez(RUN_ROOT/f'new_probe_layer_{layer_i}.npz',W=W[layer_i],bias=bias,
             active_indices=ACTIVE,train_change_counts=train_change_counts)
    for arm in ('baseline','bootstrap_equal','all'):
        for hour in (1,2):
            mask=np.ones(len(report_pos),dtype=bool)
            if arm!='all':mask &= source_arm_ids[report_pos]==arm
            mask &= source_hours[report_pos]==hour
            if not mask.any():continue
            pred=sparse[mask];truth=targets[report_pos][mask]
            truth_change=np.abs(truth)>1e-10;pred_change=pred!=0
            probe_metrics.append({'layer':layer_i,'arm':arm,'hour':hour,'examples':int(mask.sum()),
                'nonzero_target_entries':int(truth_change.sum()),
                'weighted_scaled_mae':weighted_mae(pred/scales,truth/scales),
                'change_f1':float(f1_score(truth_change.ravel(),pred_change.ravel(),zero_division=0)),
                'unchanged_exact_zero_rate':float(np.mean(~pred_change[~truth_change])),
                'changed_sign_accuracy':float(np.mean(np.sign(pred[truth_change])==
                                                   np.sign(truth[truth_change]))) if truth_change.any() else None})
    probe_controls[str(layer_i)]={'always_zero_weighted_scaled_mae':weighted_mae(
        np.zeros_like(targets[report_pos]),y_scaled[report_pos])}
probe_table=pd.DataFrame(probe_metrics)
probe_table.to_csv(RUN_ROOT/'probe_hour_environment_metrics.csv',index=False)
print('Active steerable attributes',[(int(j),VECTOR_NAMES[j],int(train_change_counts[j])) for j in ACTIVE])
print(probe_table.to_string(index=False))

def build_directions(weights):
    weights=np.asarray(weights,dtype=np.float32)
    assert weights.shape==(len(ACTIVE),) and np.all(weights>=0) and weights.sum()>0
    return {layer:normalized(U[layer]@weights) for layer in SELECTED_LAYERS}

DIRS={f'attribute_{j}':{layer:U[layer][:,k].copy() for layer in SELECTED_LAYERS}
      for k,j in enumerate(ACTIVE)}
DIRS['equal_normalized']=build_directions(np.ones(len(ACTIVE)))
activation_norms.update({layer:float(np.median(np.linalg.norm(features[train_pos,layer],axis=1)))
                         for layer in SELECTED_LAYERS})
assert all(np.isfinite(activation_norms[layer]) and activation_norms[layer]>0
           for layer in SELECTED_LAYERS)
print('New directions and activation norms ready')

## 3. Calibrate new attribute-direction weights over two-hour worlds

The calibration seeds are independent of probe-source and final seeds. Each
individual new direction is run against its paired baseline for the same
two-hour horizon as the final evaluation. Paired final-score effects are
standardized, softmax weighted, and shrunk 25% toward uniform weights. This
learns a nonnegative combination from measured simulator outcomes without
backpropagating through discrete bee choices. The calibration rule and 0.03
strength are fixed before the final seeds are evaluated.

In [ ]:
calibration=[]; cal_baselines={}
for seed in CAL_SEEDS:
    baseline,policy=run_arm(f'cal_{seed}_baseline',seed,CAL_HOURS,
                            capture_norm=(seed==CAL_SEEDS[0]))
    if seed==CAL_SEEDS[0]:
        activation_norms.update({layer:float(np.median(policy.samples[layer]))
                                 for layer in SELECTED_LAYERS})
        assert all(np.isfinite(v) and v>0 for v in activation_norms.values())
        print('Fixed activation norms',activation_norms)
    baseline['arm']='baseline'; baseline['changed_actions_vs_baseline']=0
    baseline['score_difference_vs_baseline']=0.0
    baseline['vector_difference_vs_baseline']=[0.0]*14
    cal_baselines[seed]=baseline; calibration.append(baseline)
    for arm,directions in DIRS.items():
        row,_=run_arm(f'cal_{seed}_{arm}',seed,CAL_HOURS,directions)
        row['arm']=arm; calibration.append(annotate_changes(row,baseline))
cal_table=pd.DataFrame([{k:v for k,v in row.items() if k not in ('initial_vector','final_vector','vector_difference_vs_baseline')}
                        for row in calibration])
cal_table.to_csv(RUN_ROOT/'calibration_summary.csv',index=False)
print(cal_table[['seed','arm','final_score','score_difference_vs_baseline','changed_actions_vs_baseline']].to_string(index=False))

In [ ]:
effects=np.asarray([
    np.mean([next(r for r in calibration if r['seed']==seed and r['arm']==f'attribute_{j}')
             ['score_difference_vs_baseline'] for seed in CAL_SEEDS])
    for j in ACTIVE],dtype=np.float64)
effect_scale=max(float(np.std(effects)),1e-5)
logits=2.0*(effects-effects.mean())/effect_scale
soft=np.exp(logits-logits.max()); soft/=soft.sum()
learned_weights=0.25/len(ACTIVE)+0.75*soft
assert np.all(learned_weights>0) and abs(learned_weights.sum()-1)<1e-8
DIRS['learned_combination']=build_directions(learned_weights)
print('Measured individual paired score effects',dict(zip([VECTOR_NAMES[j] for j in ACTIVE],effects.tolist())))
print('Learned weights',dict(zip([VECTOR_NAMES[j] for j in ACTIVE],learned_weights.tolist())))
for seed in CAL_SEEDS:
    row,_=run_arm(f'cal_{seed}_learned_combination',seed,CAL_HOURS,DIRS['learned_combination'])
    row['arm']='learned_combination'
    calibration.append(annotate_changes(row,cal_baselines[seed]))
print('Learned calibration scores',[(r['seed'],r['score_difference_vs_baseline'],r['changed_actions_vs_baseline'])
                                    for r in calibration if r['arm']=='learned_combination'])

## 4. Paired two-hour final evaluation on new seeds

Seeds 100–103 were not used for the probe, layer selection, or combination
weights. Each of baseline, equal-normalized steering, and outcome-weighted
steering acts once per living bee every fifteen simulated minutes for two
hours. The action order, background updates, researcher-only vector snapshots,
and local feasible menu match the previous notebook.

In [ ]:
final_runs=[]
for seed in FINAL_SEEDS:
    baseline,_=run_arm(f'final_{seed}_baseline',seed,FINAL_HOURS)
    baseline['arm']='baseline'; baseline['changed_actions_vs_baseline']=0
    baseline['score_difference_vs_baseline']=0.0
    baseline['vector_difference_vs_baseline']=[0.0]*14
    final_runs.append(baseline)
    for arm in ('equal_normalized','learned_combination'):
        directions=DIRS[arm]
        row,_=run_arm(f'final_{seed}_{arm}',seed,FINAL_HOURS,directions)
        row['arm']=arm; final_runs.append(annotate_changes(row,baseline))
print('Final arms',len(final_runs),'total actions',sum(r['actions'] for r in final_runs))

In [ ]:
def percentile_bootstrap(values,draws=10000):
    values=np.asarray(values,dtype=np.float64)
    rng=np.random.default_rng(904)
    means=rng.choice(values,size=(draws,len(values)),replace=True).mean(axis=1)
    return np.quantile(means,[0.025,0.975]).tolist()

summary_rows=[]
for row in final_runs:
    summary_rows.append({k:v for k,v in row.items()
                         if k not in ('initial_vector','final_vector','vector_difference_vs_baseline')})
summary=pd.DataFrame(summary_rows)
summary.to_csv(RUN_ROOT/'final_run_summary.csv',index=False)
vector_rows=[]
for row in final_runs:
    for j,name in enumerate(VECTOR_NAMES):
        vector_rows.append({'seed':row['seed'],'arm':row['arm'],'attribute':name,
                            'initial':row['initial_vector'][j],
                            'final':row['final_vector'][j],
                            'minus_baseline':row['vector_difference_vs_baseline'][j]})
pd.DataFrame(vector_rows).to_csv(RUN_ROOT/'final_vectors.csv',index=False)

paired={}
for arm in ('equal_normalized','learned_combination'):
    rows=[r for r in final_runs if r['arm']==arm]
    differences=[r['score_difference_vs_baseline'] for r in rows]
    paired[arm]={'score_differences':differences,
                 'mean_score_difference':float(np.mean(differences)),
                 'bootstrap_95pct_interval':percentile_bootstrap(differences),
                 'changed_actions':sum(r['changed_actions_vs_baseline'] for r in rows),
                 'mean_final_vector':np.mean([r['final_vector'] for r in rows],axis=0).tolist(),
                 'mean_vector_difference':np.mean([r['vector_difference_vs_baseline'] for r in rows],axis=0).tolist()}
report={'model':'Qwen3.5-0.8B continued bee-menu LoRA',
        'probe_source':'new mixed-environment immediate vector probe',
        'source_seeds':list(SOURCE_SEEDS),'source_hours':SOURCE_HOURS,
        'source_runs':source_summaries,
        'probe_split':{'train':len(train_pos),'layer_screen':len(screen_pos),
                       'independent_report':len(report_pos)},
        'probe_train_change_counts':dict(zip(VECTOR_NAMES,train_change_counts.tolist())),
        'probe_hour_environment_metrics':probe_metrics,
        'probe_controls':probe_controls,
        'vector_names':list(VECTOR_NAMES),'active_indices':ACTIVE.tolist(),
        'selected_layers':list(SELECTED_LAYERS),'fraction':FRACTION,'position':POSITION,
        'fixed_activation_norms':activation_norms,
        'calibration_seeds':list(CAL_SEEDS),'calibration_hours':CAL_HOURS,
        'final_seeds':list(FINAL_SEEDS),'final_hours':FINAL_HOURS,'smoke':SMOKE,
        'learned_weights':dict(zip([VECTOR_NAMES[j] for j in ACTIVE],learned_weights.tolist())),
        'individual_calibration_effects':dict(zip([VECTOR_NAMES[j] for j in ACTIVE],effects.tolist())),
        'calibration_runs':calibration,'final_runs':final_runs,'paired':paired}
(RUN_ROOT/'experiment_report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
print(summary[['seed','arm','final_score','score_difference_vs_baseline','changed_actions_vs_baseline']].to_string(index=False))
print('Paired summary',json.dumps({k:{'mean':v['mean_score_difference'],
                                     'interval':v['bootstrap_95pct_interval'],
                                     'changed_actions':v['changed_actions']}
                                    for k,v in paired.items()},indent=2))
print('Artifacts',RUN_ROOT)

## 5. Preserve every step change and overlay the three trajectories

The timestamped CSV contains every source, calibration, and final action's
immediate, background, and total change for all fourteen attributes. The
notebook saves one plot per final seed with baseline, equal-weight, and
outcome-weighted trajectories on the same axes, plus their mean score
trajectory across seeds.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

step_rows=[]
all_runs=[('source',row) for row in source_summaries]+[('calibration',row) for row in calibration]+[('final',row) for row in final_runs]
for phase,row in all_runs:
    output_dir=Path(row['output_dir'])
    actions=[json.loads(line) for line in (output_dir/'actions.jsonl').open()]
    vectors=[json.loads(line) for line in (output_dir/'vectors.jsonl').open()]
    metadata=json.loads((output_dir/'metadata.json').read_text())
    assert len(actions)==len(vectors)==row['actions'] and metadata['completed']
    start=pd.Timestamp(metadata['initial_timestamp'])
    previous=np.asarray(metadata['initial_vector'],dtype=np.float64)
    for action,vector_row in zip(actions,vectors):
        immediate=np.asarray(action['immediate_action_delta'],dtype=np.float64)
        background=np.asarray(action['background_delta'],dtype=np.float64)
        total=immediate+background
        current=np.asarray(vector_row['vector'],dtype=np.float64)
        assert immediate.shape==background.shape==total.shape==current.shape==(14,)
        assert np.allclose(previous+total,current,rtol=0,atol=1e-8)
        timestamp=pd.Timestamp(action['timestamp'])
        item={'phase':phase,'seed':row['seed'],'arm':row['arm'],
              'event_id':action['event_id'],'bee_id':action['bee_id'],
              'timestamp':action['timestamp'],
              'sim_hour':(timestamp-start).total_seconds()/3600.0,
              'action':action['action']['action'],
              'outcome_status':action['outcome']['status'],
              'total_score_delta':float(total.mean()),
              'immediate_score_delta':float(immediate.mean()),
              'background_score_delta':float(background.mean())}
        for j,name in enumerate(VECTOR_NAMES):
            item[f'total_{name}']=float(total[j])
            item[f'immediate_{name}']=float(immediate[j])
            item[f'background_{name}']=float(background[j])
            item[f'value_{name}']=float(current[j])
        step_rows.append(item)
        previous=current
    assert np.allclose(previous,row['final_vector'],rtol=0,atol=1e-8)

steps=pd.DataFrame(step_rows)
assert len(steps)==sum(row['actions'] for _,row in all_runs)
steps['hive_score']=steps[[f'value_{name}' for name in VECTOR_NAMES]].mean(axis=1)
step_file=RUN_ROOT/'all_step_changes.csv.gz'
steps.to_csv(step_file,index=False,compression='gzip')
print('Saved',len(steps),'step-change records across all 14 attributes to',step_file)
print('Columns include total_, immediate_, background_, and value_ for each attribute')
print(steps[['phase','seed','arm','event_id','timestamp','sim_hour',
             'total_adult_energy_sufficiency','immediate_adult_energy_sufficiency',
             'background_adult_energy_sufficiency']].head(5).to_string(index=False))

PLOT_ARMS=('baseline','equal_normalized','learned_combination')
assert all(arm=='baseline' or arm in DIRS for arm in PLOT_ARMS)
colors={'baseline':'#555555','equal_normalized':'#087e8b',
        'learned_combination':'#c17c00'}
plot_paths=[]
for seed in FINAL_SEEDS:
    subset=steps[(steps.phase=='final') & (steps.seed==seed)]
    fig,axes=plt.subplots(14,1,sharex=True,figsize=(14,27),constrained_layout=True)
    for j,name in enumerate(VECTOR_NAMES):
        ax=axes[j]
        for arm in PLOT_ARMS:
            part=subset[subset.arm==arm]
            changed=part[np.abs(part[f'total_{name}'])>1e-10]
            ax.scatter(changed.sim_hour,changed[f'total_{name}'],
                       s=9,alpha=0.55,label=arm if j==0 else None,
                       color=colors.get(arm,None))
        ax.axhline(0,color='black',linewidth=0.5,alpha=0.4)
        ax.set_ylabel(name,fontsize=7)
        ax.grid(axis='x',alpha=0.2)
    axes[0].legend(loc='upper right',fontsize=8,ncol=3)
    axes[-1].set_xlabel('Simulated hours')
    axes[-1].set_xlim(0,FINAL_HOURS)
    fig.suptitle(f'Per-action hive attribute step changes — seed {seed}')
    path=RUN_ROOT/f'step_changes_seed_{seed}.png'
    fig.savefig(path,dpi=130)
    plot_paths.append(str(path))
    if seed==FINAL_SEEDS[0]: display(fig)
    plt.close(fig)

    fig,axes=plt.subplots(14,1,sharex=True,figsize=(14,27),constrained_layout=True)
    for j,name in enumerate(VECTOR_NAMES):
        ax=axes[j]
        for arm in PLOT_ARMS:
            part=subset[subset.arm==arm]
            ax.plot(part.sim_hour,part[f'value_{name}'],
                    linewidth=0.85,label=arm if j==0 else None,
                    color=colors.get(arm,None))
        ax.set_ylabel(name,fontsize=7)
        ax.set_ylim(0,1)
        ax.grid(alpha=0.2)
    axes[0].legend(loc='upper right',fontsize=8,ncol=3)
    axes[-1].set_xlabel('Simulated hours')
    axes[-1].set_xlim(0,FINAL_HOURS)
    fig.suptitle(f'Hive state vector trajectories — seed {seed}')
    path=RUN_ROOT/f'state_trajectories_seed_{seed}.png'
    fig.savefig(path,dpi=130)
    plot_paths.append(str(path))
    if seed==FINAL_SEEDS[0]: display(fig)
    plt.close(fig)

    fig,ax=plt.subplots(figsize=(13,5),constrained_layout=True)
    initial_score=next(r['initial_score'] for r in final_runs
                       if r['seed']==seed and r['arm']=='baseline')
    for arm in PLOT_ARMS:
        part=subset[subset.arm==arm]
        times=np.r_[0.0,part.sim_hour.to_numpy()]
        scores=np.r_[initial_score,part.hive_score.to_numpy()]
        ax.plot(times,scores,label=arm,color=colors.get(arm,None),linewidth=1.2)
    ax.set(xlabel='Simulated hours',ylabel='Mean of 14 hive attributes',
           title=f'Hive score trajectories — seed {seed}',xlim=(0,FINAL_HOURS))
    ax.grid(alpha=0.25); ax.legend()
    path=RUN_ROOT/f'score_trajectories_seed_{seed}.png'
    fig.savefig(path,dpi=150)
    plot_paths.append(str(path))
    if seed==FINAL_SEEDS[0]: display(fig)
    plt.close(fig)

fig,ax=plt.subplots(figsize=(13,5),constrained_layout=True)
for arm in PLOT_ARMS:
    part=steps[(steps.phase=='final') & (steps.arm==arm)]
    average=part.groupby('event_id',sort=True)[['sim_hour','hive_score']].mean()
    ax.plot(np.r_[0.0,average.sim_hour.to_numpy()],
            np.r_[report['final_runs'][0]['initial_score'],average.hive_score.to_numpy()],
            label=arm,color=colors.get(arm,None),linewidth=1.4)
ax.set(xlabel='Simulated hours',ylabel='Mean hive score across final seeds',
       title='Baseline, equal-weight, and learned-weight score trajectories',
       xlim=(0,FINAL_HOURS))
ax.grid(alpha=0.25); ax.legend()
path=RUN_ROOT/'mean_score_trajectories.png'
fig.savefig(path,dpi=160)
plot_paths.append(str(path))
display(fig)
plt.close(fig)

report['step_change_records']=len(steps)
report['step_change_csv']=str(step_file)
report['step_change_plot_arms']=list(PLOT_ARMS)
report['step_change_plots']=plot_paths
(RUN_ROOT/'experiment_report.json').write_text(json.dumps(report,indent=2,allow_nan=False))
print('Saved',len(plot_paths),'figures; step changes and trajectories are available for every final seed')